# <font color="Green">**Notebook Purpose**</font>

This notebook generates three complementary cluster-level visualizations that summarize treatment behavior and clinical changes across time. First, it produces a cluster-level heatmap that displays the individual prescription sequences of all patients within the cluster. Second, it generates medication distribution curves that show the percentage of patients taking each drug class in every semiannual bin from 2019 to 2024. Third, it creates trajectory plots that depict changes in mean BMI and mean HbA1c for each cluster between 2019 and 2024, with optional highlighting of a selected cluster.

These three components are then combined into a single cluster-level summary figure, and the notebook provides functionality for exporting ranked sets of these figures as grouped PDF files. Together, these outputs offer a multi-angle view of treatment patterns, medication adoption, and clinical shifts within each cluster and across therapy groups.



---

### <font color="Red">Required Data</font>

The following preprocessed objects must be present in the working directory before running this notebook.  
Examples showing how these objects are created are available in other notebooks within the **Data Preprocessing** folder.


#### **1. `patient_bins.pkl`**  
A Python dictionary mapping each `patient_id` to a list of 12 prescription bins.

Format:  
- **Keys:** patient IDs  
- **Values:** lists of 12 semiannual bins  
  - Each bin is a list of prescription class strings  
    (e.g., `["MET"]`, `["SGLT2", "MET"]`, `["Insulin"]`)


#### **2. `sorted_cluster_patient_ids.pkl`**  
A dictionary mapping each cluster ID to the *ordered* list of patient IDs in that cluster.

Format:  
- **Keys:** cluster IDs  
- **Values:** ordered lists of patient IDs  
  This ordering is used directly for cluster-level aggregation.


#### **3. `BMI_vital_signs` dataframe**  
A patient-level dataframe containing all recorded BMI measurements.

Must include:  
- `patient_id`  
- `date`  
- `value` (BMI value)

Used to compute mean BMI in the 2019 and 2024 windows for each cluster.


#### **4. `lab_results` dataframe (HbA1c)**  
A patient-level dataframe containing HbA1c laboratory measurements.

Must include:  
- `patient_id`  
- `date`  
- `lab_result_num_val` (HbA1c value)

Used to compute mean HbA1c in the 2019 and 2024 windows for each cluster and to assess glycemic control.


All data objects must reference the same underlying patient cohort to ensure consistent mapping between cluster membership, prescription bins, BMI data, and HbA1c measurements.



In [ ]:
import pandas as pd
import numpy as np
import pickle

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib.gridspec as gridspec
from collections import OrderedDict

import statsmodels.formula.api as smf

##<font color="black">**Read in Data**</font>

In [ ]:
with open('/content/sorted_cluster_patient_ids.pkl', 'rb') as f:
    sorted_cluster_patient_ids = pickle.load(f)

with open('/content/patient_bins.pkl', 'rb') as f:
    patient_bins = pickle.load(f)

BMI_mixed_effect = pd.read_csv('/content/BMI_by_cluster_by_year.csv')

HbA1c_mixed_effect = pd.read_csv('/content/HbA1c_by_cluster_by_year.csv')

In [ ]:
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
patient_demographics.head()

##<font color="black">**Functions for Each Plots Creation**</font>

In [ ]:
# Prescription colors - used in two of the plots
PRESCRIPTION_COLORS = {
    'SUL': '#FFC0CB',       # Light pink
    'SGLT2': '#008000',     # Green
    'Insulin': '#FF0000',   # Red
    'MET': '#0000FF',       # Blue
    'DPP-4': '#8B4513',     # Brown
    'GLP-1': '#FFDB58',     # Mustard
    'GIP/GLP-1': '#40E0D0', # Turquoise
    'TZD': '#FF8C00',       # Bright Orange (DarkOrange)
    'Other': '#000000',     # Black
    'nothing': '#FFFFFF'    # White (no prescription)
}

###<font color="black">**Cluster-level Patient Medication Heatmap**</font>

In [ ]:
# This function creates the mixing effect (e.g. MET (Blue) + Insulin (Red) --> Purple)
def mix_colors(colors):
    rgb_colors = np.array([to_rgb(PRESCRIPTION_COLORS[color]) for color in colors if color in PRESCRIPTION_COLORS])
    if len(rgb_colors) == 0:
        return np.array(to_rgb(PRESCRIPTION_COLORS['nothing']))
    mixed_rgb = np.mean(rgb_colors, axis=0)
    return np.array(mixed_rgb, dtype=np.float32)

def plot_cluster_heatmap(ax, cluster_patient_ids, patient_bins_cluster):
    num_bins = 12
    heatmap_data = np.ones((len(cluster_patient_ids), num_bins, 3), dtype=np.float32)

    for i, patient_id in enumerate(cluster_patient_ids):
        for j, prescriptions in enumerate(patient_bins_cluster[patient_id]):
            heatmap_data[i, j] = mix_colors(prescriptions)

    ax.imshow(heatmap_data, aspect='auto', interpolation='none')
    ax.set_xticks(range(num_bins))

    # Label only H1 for each year, leave others blank
    labels = [f'{year}' if h == 1 else '' for year in range(2019, 2025) for h in (1, 2)]
    ax.set_xticklabels(labels, rotation=45, ha='right')

    ax.set_yticks([])
    ax.set_title("1. Individual Medication Sequences", fontsize=14)
    ax.font_size=15

###<font color="black">**Cluster-level Medication Distribution Plot**</font>

In [ ]:
def plot_medication_distribution(ax, cluster_patient_ids, patient_bins_cluster):
    num_bins = 12
    drug_classes = [d for d in PRESCRIPTION_COLORS.keys() if d != 'nothing']

    drug_counts = {drug: [0]*num_bins for drug in drug_classes}
    total_patients = len(cluster_patient_ids)

    for patient_id in cluster_patient_ids:
        for bin_index, prescriptions in enumerate(patient_bins_cluster[patient_id]):
            for drug in prescriptions:
                if drug in drug_counts:
                    drug_counts[drug][bin_index] += 1

    for drug in drug_counts:
        drug_counts[drug] = [count / total_patients * 100 for count in drug_counts[drug]]

    for drug, percentages in drug_counts.items():
        ax.plot(range(num_bins), percentages, label=drug, color=PRESCRIPTION_COLORS[drug])

    ax.set_ylim(0, 100)
    ax.set_xticks(range(num_bins))

    # Label only H1 for each year
    labels = [f'{year}' if h == 1 else '' for year in range(2019, 2025) for h in (1, 2)]
    ax.set_xticklabels(labels, rotation=45, ha='right')

    ax.set_title("2. % Taking Each Medication Class", fontsize=14)
    ax.grid(True)
    ax.set_ylabel("")
    ax.legend().remove()

###<font color="black">**Cluster-level BMI vs HbA1c Plot**</font>

In [ ]:
def compute_age_adjusted_trends(lab_results, BMI_vital_signs,
                                patient_demographics, cluster_patient_ids):
    """
    Returns:
      - bmi_results_df: age-adjusted BMI means + 95% CI at 2019–2024
      - a1c_results_df: age-adjusted HbA1c means + 95% CI at 2019–2024
      - cluster_counts: dict[cluster_id] = {
            "cluster_total": ...,
            "bmi_used": ...,
            "a1c_used": ...
        }

    Eligibility rule for each measure (BMI / HbA1c):
      - A patient is included ONLY if they have at least one measurement in
        *every* calendar year from 2019 through 2024 (all 6 years).

    Modeling:
      - For BMI and HbA1c separately, we fit:
            value ~ C(year) + age
        where year is 2019–2024 (categorical) and age = year - year_of_birth.
      - We then predict adjusted means for each year, holding age at the mean.
    """

    # Copies + datetime
    bmi_df = BMI_vital_signs.copy()
    bmi_df["date"] = pd.to_datetime(bmi_df["date"])

    lab_df = lab_results.copy()
    lab_df["date"] = pd.to_datetime(lab_df["date"])

    # Demographics for age
    demo = patient_demographics[["patient_id", "year_of_birth"]].dropna()

    # Years to require for "complete" trajectories
    required_years = list(range(2019, 2025))  # [2019, 2020, 2021, 2022, 2023, 2024]

    bmi_results = []
    a1c_results = []
    cluster_counts = {}

    for cluster_id, patients in cluster_patient_ids.items():
        cluster_total = len(patients)
        bmi_used_patients = set()
        a1c_used_patients = set()

        # =====================================================================
        # BMI: require data in ALL 6 years, then model with C(year) + age
        # =====================================================================
        bmi_cluster = bmi_df[bmi_df["patient_id"].isin(patients)].copy()
        if not bmi_cluster.empty:
            bmi_cluster["year"] = bmi_cluster["date"].dt.year
            bmi_cluster = bmi_cluster[bmi_cluster["year"].between(2019, 2024)]

            # Per-patient, per-year mean BMI
            bmi_yearly = (
                bmi_cluster
                .groupby(["patient_id", "year"])["value"]
                .mean()
                .unstack("year")
            )

            if not bmi_yearly.empty:
                # Only keep patients with values in EACH required year
                bmi_complete = bmi_yearly.dropna(subset=required_years, how="any")
            else:
                bmi_complete = pd.DataFrame()

            if not bmi_complete.empty:
                # Long format: one row per patient-year
                bmi_long = (
                    bmi_complete
                    .reset_index()
                    .melt(
                        id_vars="patient_id",
                        value_vars=required_years,
                        var_name="year",
                        value_name="value"
                    )
                )

                # Merge with demographics
                bmi_long = bmi_long.merge(demo, on="patient_id", how="inner")

                if not bmi_long.empty:
                    # Only count patients that survive the merge (i.e., have YOB)
                    bmi_used_patients.update(bmi_long["patient_id"].unique())

                    # Age & keep year as int
                    bmi_long["year"] = bmi_long["year"].astype(int)
                    bmi_long["age"] = bmi_long["year"] - bmi_long["year_of_birth"]

                    try:
                        # Categorical year effect + age adjustment
                        model_bmi = smf.ols("value ~ C(year) + age",
                                            data=bmi_long).fit()
                        mean_age = bmi_long["age"].mean()

                        # Predict adjusted means for each year at mean_age
                        pred_df = pd.DataFrame({
                            "year": required_years,
                            "age": [mean_age] * len(required_years),
                        })
                        pred = model_bmi.get_prediction(pred_df)\
                                        .summary_frame(alpha=0.05)

                        for i, year in enumerate(required_years):
                            bmi_results.append({
                                "cluster": cluster_id,
                                "time": str(year),
                                "mean": pred.loc[i, "mean"],
                                "ci_lower": pred.loc[i, "mean_ci_lower"],
                                "ci_upper": pred.loc[i, "mean_ci_upper"],
                            })
                    except Exception:
                        # Fallback: unadjusted mean + 95% CI per year
                        for year in required_years:
                            vals = bmi_complete[year].values
                            m = np.mean(vals)
                            se = np.std(vals, ddof=1) / np.sqrt(len(vals))
                            bmi_results.append({
                                "cluster": cluster_id,
                                "time": str(year),
                                "mean": m,
                                "ci_lower": m - 1.96 * se,
                                "ci_upper": m + 1.96 * se,
                            })

        # =====================================================================
        # HbA1c: require data in ALL 6 years, then model with C(year) + age
        # =====================================================================
        a1c_cluster = lab_df[lab_df["patient_id"].isin(patients)].copy()
        if not a1c_cluster.empty:
            a1c_cluster["year"] = a1c_cluster["date"].dt.year
            a1c_cluster = a1c_cluster[a1c_cluster["year"].between(2019, 2024)]

            a1c_yearly = (
                a1c_cluster
                .groupby(["patient_id", "year"])["lab_result_num_val"]
                .mean()
                .unstack("year")
            )

            if not a1c_yearly.empty:
                a1c_complete = a1c_yearly.dropna(subset=required_years, how="any")
            else:
                a1c_complete = pd.DataFrame()

            if not a1c_complete.empty:
                a1c_long = (
                    a1c_complete
                    .reset_index()
                    .melt(
                        id_vars="patient_id",
                        value_vars=required_years,
                        var_name="year",
                        value_name="value"
                    )
                )

                a1c_long = a1c_long.merge(demo, on="patient_id", how="inner")

                if not a1c_long.empty:
                    a1c_used_patients.update(a1c_long["patient_id"].unique())

                    a1c_long["year"] = a1c_long["year"].astype(int)
                    a1c_long["age"] = a1c_long["year"] - a1c_long["year_of_birth"]

                    try:
                        model_a1c = smf.ols("value ~ C(year) + age",
                                            data=a1c_long).fit()
                        mean_age = a1c_long["age"].mean()

                        pred_df = pd.DataFrame({
                            "year": required_years,
                            "age": [mean_age] * len(required_years),
                        })
                        pred = model_a1c.get_prediction(pred_df)\
                                        .summary_frame(alpha=0.05)

                        for i, year in enumerate(required_years):
                            a1c_results.append({
                                "cluster": cluster_id,
                                "time": str(year),
                                "mean": pred.loc[i, "mean"],
                                "ci_lower": pred.loc[i, "mean_ci_lower"],
                                "ci_upper": pred.loc[i, "mean_ci_upper"],
                            })
                    except Exception:
                        # Fallback: unadjusted mean + 95% CI per year
                        for year in required_years:
                            vals = a1c_complete[year].values
                            m = np.mean(vals)
                            se = np.std(vals, ddof=1) / np.sqrt(len(vals))
                            a1c_results.append({
                                "cluster": cluster_id,
                                "time": str(year),
                                "mean": m,
                                "ci_lower": m - 1.96 * se,
                                "ci_upper": m + 1.96 * se,
                            })

        # =====================================================================
        # Record counts for this cluster
        # =====================================================================
        cluster_counts[cluster_id] = {
            "cluster_total": cluster_total,
            "bmi_used": len(bmi_used_patients),
            "a1c_used": len(a1c_used_patients),
        }

    bmi_results_df = pd.DataFrame(bmi_results)
    a1c_results_df = pd.DataFrame(a1c_results)

    return bmi_results_df, a1c_results_df, cluster_counts

In [ ]:
bmi_adj, a1c_adj, cluster_counts = compute_age_adjusted_trends(
    lab_results=lab_results,
    BMI_vital_signs=BMI_vital_signs,
    patient_demographics=patient_demographics,
    cluster_patient_ids=sorted_cluster_patient_ids  # or cluster_patient_ids
)

In [ ]:
def plot_age_adjusted_cluster_panel(
    ax,
    cluster_id,
    BMI_mixed_effect,
    HbA1c_mixed_effect,
    cluster_counts=None,          # optional: only used for N=...
    bmi_ylim=(27.5, 40),
    a1c_ylim=(6, 11),
):
    """
    Panel 3: Adjusted BMI & HbA1c trajectories (2019–2024) using mixed-effect outputs.

    Required columns in each df:
      - cluster_id, year_f, emmean, lower.CL, upper.CL
    """
    import numpy as np

    bmi_color = "#1f77b4"   # blue
    a1c_color = "#d95f02"   # dark orange

    REQUIRED_COLS = {"cluster_id", "year_f", "emmean", "lower.CL", "upper.CL"}

    def _prep(df):
        if df is None or df.empty:
            return df

        missing = REQUIRED_COLS - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")

        sub = df.loc[df["cluster_id"] == cluster_id, ["year_f", "emmean", "lower.CL", "upper.CL"]].copy()

        # parse year as int from year_f (handles "2019", "2019H1", etc.)
        sub["year_int"] = (
            sub["year_f"].astype(str)
            .str.extract(r"(\d{4})", expand=False)
            .astype(float)
        )

        sub = sub.dropna(subset=["year_int"])
        sub["year_int"] = sub["year_int"].astype(int)
        sub = sub.sort_values("year_int")

        return sub

    def _plot_series(ax_, sub, color, marker, linestyle):
        if sub is None or sub.empty:
            return

        x = np.array([x_positions[y] for y in sub["year_int"]])
        y = sub["emmean"].to_numpy()

        lower = y - sub["lower.CL"].to_numpy()
        upper = sub["upper.CL"].to_numpy() - y
        yerr = np.vstack([lower, upper])

        ax_.errorbar(
            x, y, yerr=yerr,
            marker=marker, linestyle=linestyle, capsize=3,
            color=color
        )

    # ---- subset BMI + HbA1c for this cluster_id
    sub_bmi = _prep(BMI_mixed_effect)
    sub_a1c = _prep(HbA1c_mixed_effect)

    # ---- if both empty, show message and exit
    if (sub_bmi is None or sub_bmi.empty) and (sub_a1c is None or sub_a1c.empty):
        ax.text(
            0.5, 0.5,
            "No adjusted BMI/HbA1c rows for this cluster",
            ha="center", va="center", fontsize=10
        )
        ax.set_title("3. Adjusted BMI & HbA1c", fontsize=14, pad=8)
        ax.grid(False)
        return

    # ---- shared x-axis based on union of years present
    years = sorted(set(sub_bmi["year_int"]).union(set(sub_a1c["year_int"])))
    x_positions = {yr: i for i, yr in enumerate(years)}
    ax.set_xticks(np.arange(len(years)))
    ax.set_xticklabels([str(y) for y in years])

    # ---- BMI on left
    _plot_series(ax, sub_bmi, bmi_color, marker="o", linestyle="-")
    ax.grid(True, alpha=0.3)

    if bmi_ylim is not None:
        ax.set_ylim(*bmi_ylim)
        if bmi_ylim == (27.5, 40):
            ax.set_yticks(np.arange(27.5, 40.1, 2.5))

    ax.set_ylabel("BMI", color=bmi_color, fontsize=13)
    ax.tick_params(axis="y", colors=bmi_color)
    ax.spines["left"].set_color(bmi_color)

    # ---- HbA1c on right
    ax2 = ax.twinx()
    _plot_series(ax2, sub_a1c, a1c_color, marker="s", linestyle="--")

    if a1c_ylim is not None:
        ax2.set_ylim(*a1c_ylim)
        if a1c_ylim == (6, 11):
            ax2.set_yticks(np.arange(6, 11.1, 1))

    ax2.set_ylabel("HbA1c (%)", color=a1c_color, fontsize=13)
    ax2.tick_params(axis="y", colors=a1c_color)
    ax2.spines["right"].set_color("black")

    # ---- title (set once)
    title = "3. Adjusted BMI & HbA1c"
    if cluster_counts is not None and cluster_id in cluster_counts:
        n = cluster_counts[cluster_id].get("cluster_total", "?")
        title = f"{title} (N={n})"
    ax.set_title(title, fontsize=14, pad=8)


##<font color="black">**Combined Cluster-Level Visualizations and PDF Export**</font>

In [ ]:
"""
This block creates a three-part summary figure for every cluster, groups clusters into their
therapy categories, and ranks them by size. The resulting structure is used later to export
organized cluster-level PDFs.
"""

from collections import OrderedDict
import matplotlib.pyplot as plt

# ---------- 1) Build one figure per cluster_id ----------
def build_cluster_figure(
    cluster_id,
    patient_ids,
    BMI_mixed_effect,
    HbA1c_mixed_effect,
):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    patient_bins_cluster = {pid: patient_bins[pid] for pid in patient_ids}

    # Panel 1: heatmap of individual trajectories
    plot_cluster_heatmap(axes[0], patient_ids, patient_bins_cluster)

    # Panel 2: medication distribution curves
    plot_medication_distribution(axes[1], patient_ids, patient_bins_cluster)

    # Panel 3: adjusted BMI & HbA1c trajectories (2019–2024) on twin axes
    plot_age_adjusted_cluster_panel(
        ax=axes[2],
        cluster_id=cluster_id,
        BMI_mixed_effect=BMI_mixed_effect,
        HbA1c_mixed_effect=HbA1c_mixed_effect,
        bmi_ylim=(27.5, 40),
        a1c_ylim=(6, 11),
    )

    # Give slightly more right margin so the right y-label (HbA1c) is not cut off
    fig.subplots_adjust(left=0.05, right=0.94, top=0.88, bottom=0.12, wspace=0.18)
    return fig


# Build all once, keyed by cluster_id
cluster_figures = {}
for cluster_id, pids in sorted_cluster_patient_ids.items():
    fig = build_cluster_figure(
        cluster_id=cluster_id,
        patient_ids=pids,
        BMI_mixed_effect=BMI_mixed_effect,
        HbA1c_mixed_effect=HbA1c_mixed_effect,
    )
    cluster_figures[cluster_id] = fig
    plt.close(fig)  # keep memory tidy—the fig is still in dict


# ---------- 2) Define your groups ----------
groups = {
    "Early Dropout Group": [1, 2, 3, 4, 5, 9, 11, 15, 17, 19],
    "Variant Therapy":     [13, 16, 21],
    "Monotherapy":         [6, 7, 8, 18, 29],
    "Dual Therapy":        [12, 23, 25, 28, 31, 32, 36, 37],
    "Complex Therapy":     [22, 34, 40],
    "GLP-1 Therapy":       [10, 14, 20, 24, 26, 27, 30, 33, 35, 38, 39],
}


# ---------- 3) Build nested dict: group -> OrderedDict(rank -> {cluster_id, fig, n_patients}) ----------
group_figures = {}
for gname, cluster_ids in groups.items():
    # rank clusters by size within the group (desc)
    ordered = sorted(
        cluster_ids,
        key=lambda cid: len(sorted_cluster_patient_ids[cid]),
        reverse=True
    )

    od = OrderedDict()
    for rank, cid in enumerate(ordered, start=1):
        od[rank] = {
            "cluster_id": cid,
            "fig": cluster_figures[cid],
            "n_patients": len(sorted_cluster_patient_ids[cid]),
        }
    group_figures[gname] = od


The export_group_pdf function is responsible for producing the grouped cluster visualizations. It allows you to specify which clusters to include, enabling flexible selection and the creation of publication-ready figures with proper sizing.

In [ ]:
def export_group_pdf(group_name, out_pdf_path=None, clusters=None):
    """
    Export ranked cluster figures for a group into a combined PDF.

    Parameters
    ----------
    group_name : str
        Name of the group in group_figures.

    out_pdf_path : str or None
        If provided, saves to PDF. Otherwise displays inline.

    clusters : list of int or None
        The *rank numbers* (within-group) to include.
        If passed, these exact numbers will appear in the visual
        instead of being renumbered 1..N.
    """

    od = group_figures[group_name]  # OrderedDict: rank -> metadata

    # If clusters list is passed, filter using those exact ranks
    if clusters is not None:
        # Preserve the order given by the user
        od = {rank: od[rank] for rank in clusters if rank in od}

    n = len(od)
    fig, axes = plt.subplots(n + 1, 1, figsize=(18, 4 * (n + 1)))
    if n + 1 == 1:
        axes = [axes]

    # Title-free spacer row at top
    axes[0].axis('off')

    # Render each selected cluster figure
    for plot_index, (rank_number, meta) in enumerate(od.items(), start=1):
        ax = axes[plot_index]
        cluster_fig = meta["fig"]
        n_pat = meta["n_patients"]

        # Convert stored figure into an RGBA image
        canvas = FigureCanvas(cluster_fig)
        canvas.draw()
        w, h = canvas.get_width_height()
        image = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)

        ax.imshow(image)
        ax.axis('off')

        # IMPORTANT CHANGE:
        # Instead of naming them "Cluster 1,2,3,...", use actual rank numbers.
        ax.text(
            0.05, 1.025,
            f"Cluster {rank_number} - {n_pat} patients:",
            transform=ax.transAxes,
            ha='left', va='bottom',
            fontsize=16,
            fontweight='bold'
        )

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.2)

    # Save output
    if out_pdf_path:
        fig.savefig(out_pdf_path, dpi=600, bbox_inches='tight', pad_inches=0.3)
        plt.close(fig)
    else:
        plt.show()

In [ ]:
# Example of calling the export function with no output path.
export_group_pdf('Early Dropout Group')

###<font color="black">**Complex & Variant Therapy**</font>

There are only three clusters in each of these groups, so I will combine them into a single visual.

In [ ]:
def export_two_groups_combined_pdf(
    group_a, group_b,
    out_pdf_path=None,
    spacer_rows=1,          # number of blank rows between groups
    top_margin_rows=1,      # blank rows at the very top
    spacer_ratio=0.15,      # relative height of each spacer row (vs cluster rows)
    top_margin_ratio=0.5    # relative height of each top-margin row
):
    """
    Combine two group visualizations into a single stacked figure.

    Uses:
        group_figures[group_name][rank] -> {
            'fig': figure,
            'n_patients': int,
            'cluster_id': original_cluster_id
        }

    Parameters
    ----------
    group_a : str
        Name of the first group in group_figures (e.g., "Complex Therapy").
    group_b : str
        Name of the second group in group_figures (e.g., "Variant Therapy").

    out_pdf_path : str or None
        If provided, saves to a file (e.g., .pdf, .tiff, .png).
        If None, displays inline.

    spacer_rows : int
        Number of blank rows between the two groups.

    top_margin_rows : int
        Number of blank rows at the very top.

    spacer_ratio : float
        Relative height of each spacer row compared to a cluster row.

    top_margin_ratio : float
        Relative height of each top-margin row compared to a cluster row.
    """

    # Pull ordered dicts for the two groups: rank -> metadata
    od_a = group_figures[group_a]
    od_b = group_figures[group_b]

    n_a = len(od_a)
    n_b = len(od_b)

    # Require at least one cluster overall
    if (n_a + n_b) == 0:
        raise ValueError("No clusters found for the two groups.")

    # Total rows = top margin + group A + spacer + group B
    total_rows = top_margin_rows + n_a + spacer_rows + n_b

    # Figure height scales with actual clusters; spacer/top rows are given
    # smaller height via height_ratios.
    fig, axes = plt.subplots(
        total_rows, 1,
        figsize=(18, 4 * (n_a + n_b)),
        gridspec_kw={
            "height_ratios": (
                [top_margin_ratio] * top_margin_rows +
                [1]               * n_a +
                [spacer_ratio]    * spacer_rows +
                [1]               * n_b
            )
        }
    )

    if total_rows == 1:
        axes = [axes]

    # Top margin rows: blank
    for i in range(top_margin_rows):
        axes[i].axis('off')

    cur = top_margin_rows

    # ----- Render Group A -----
    for rank_number, meta in od_a.items():
        ax = axes[cur]
        cur += 1

        cluster_fig = meta["fig"]
        n_pat = meta["n_patients"]

        # Render stored fig to RGBA image and embed
        canvas = FigureCanvas(cluster_fig)
        canvas.draw()
        w, h = canvas.get_width_height()
        image = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)

        ax.imshow(image)
        ax.axis('off')
        ax.text(
            0.05, 1.025,
            f"Cluster {rank_number} - {n_pat} patients:",
            transform=ax.transAxes,
            ha='left', va='bottom',
            fontsize=16,
            fontweight='bold'
        )

    # ----- Spacer rows (blank) -----
    for _ in range(spacer_rows):
        axes[cur].axis('off')
        cur += 1

    # ----- Render Group B -----
    for rank_number, meta in od_b.items():
        ax = axes[cur]
        cur += 1

        cluster_fig = meta["fig"]
        n_pat = meta["n_patients"]

        canvas = FigureCanvas(cluster_fig)
        canvas.draw()
        w, h = canvas.get_width_height()
        image = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)

        ax.imshow(image)
        ax.axis('off')
        ax.text(
            0.05, 1.025,
            f"Cluster {rank_number} - {n_pat} patients:",
            transform=ax.transAxes,
            ha='left', va='bottom',
            fontsize=16,
            fontweight='bold'
        )

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.24)

    if out_pdf_path:
        fig.savefig(out_pdf_path, dpi=600, bbox_inches='tight', pad_inches=0.3)
        plt.close(fig)
    else:
        plt.show()


In [ ]:
export_two_groups_combined_pdf(
    "Complex Therapy", "Variant Therapy",
    out_pdf_path="Complex_Therapy_Variant_TherapyNew.pdf",
    spacer_rows=1
)


###<font color="black">**External Patients Export**</font>

In [ ]:
# ============================================================
# BLOCK 1 — Two-panel figure builder + group_figures_2panel
# (Use this for Early Dropout Group: only heatmap + med curves)
# ============================================================

from collections import OrderedDict
import matplotlib.pyplot as plt

def build_cluster_figure_2panel(cluster_id, patient_ids):
    """
    Build a 2-panel cluster figure:
      Panel 1: heatmap of individual trajectories
      Panel 2: medication distribution curves
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    patient_bins_cluster = {pid: patient_bins[pid] for pid in patient_ids}

    # Panel 1
    plot_cluster_heatmap(axes[0], patient_ids, patient_bins_cluster)

    # Panel 2
    plot_medication_distribution(axes[1], patient_ids, patient_bins_cluster)

    fig.subplots_adjust(left=0.05, right=0.97, top=0.88, bottom=0.12, wspace=0.18)
    return fig


# ---- Build only the Early Dropout Group figures (2-panel) ----
EARLY_DROPOUT_CLUSTER_IDS = [1, 2, 3, 4, 5, 9, 11, 15, 17, 19]

cluster_figures_2panel = {}
for cid in EARLY_DROPOUT_CLUSTER_IDS:
    pids = sorted_cluster_patient_ids[cid]
    fig = build_cluster_figure_2panel(cluster_id=cid, patient_ids=pids)
    cluster_figures_2panel[cid] = fig
    plt.close(fig)

# ---- Rank within group by size (desc) and store like before: rank -> {cluster_id, fig, n_patients} ----
ordered = sorted(
    EARLY_DROPOUT_CLUSTER_IDS,
    key=lambda cid: len(sorted_cluster_patient_ids[cid]),
    reverse=True
)

group_figures_2panel = OrderedDict()
for rank, cid in enumerate(ordered, start=1):
    group_figures_2panel[rank] = {
        "cluster_id": cid,
        "fig": cluster_figures_2panel[cid],
        "n_patients": len(sorted_cluster_patient_ids[cid]),
    }


In [ ]:
# ============================================================
# BLOCK 2 — Export PDF for the 2-panel Early Dropout group
# (Mirrors your export_group_pdf, but uses group_figures_2panel)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

def export_group_pdf_2panel(out_pdf_path=None, clusters=None):
    """
    Export ranked 2-panel cluster figures (Early Dropout Group) into a combined PDF.

    Parameters
    ----------
    out_pdf_path : str or None
        If provided, saves to PDF. Otherwise displays inline.

    clusters : list of int or None
        The *rank numbers* (within-group) to include.
        If passed, these exact numbers will appear in the visual
        instead of being renumbered 1..N.
    """
    od = group_figures_2panel  # OrderedDict: rank -> metadata

    # If clusters list is passed, filter using those exact ranks
    if clusters is not None:
        od = {rank: od[rank] for rank in clusters if rank in od}

    n = len(od)
    if n == 0:
        raise ValueError("No clusters selected for export.")

    fig, axes = plt.subplots(n + 1, 1, figsize=(12, 4 * (n + 1)))
    if n + 1 == 1:
        axes = [axes]

    # Title-free spacer row at top
    axes[0].axis('off')

    for plot_index, (rank_number, meta) in enumerate(od.items(), start=1):
        ax = axes[plot_index]
        cluster_fig = meta["fig"]
        n_pat = meta["n_patients"]

        canvas = FigureCanvas(cluster_fig)
        canvas.draw()
        w, h = canvas.get_width_height()
        image = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)

        ax.imshow(image)
        ax.axis('off')
        ax.text(
            0.05, 1.025,
            f"Cluster {rank_number} - {n_pat} patients:",
            transform=ax.transAxes,
            ha='left', va='bottom',
            fontsize=16,
            fontweight='bold'
        )

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.2)

    if out_pdf_path:
        fig.savefig(out_pdf_path, dpi=600, bbox_inches='tight', pad_inches=0.3)
        plt.close(fig)
    else:
        plt.show()


In [ ]:
export_group_pdf_2panel(
    out_pdf_path="External_Patients_Unlabeled.pdf"
)


##<font color="black">**Creating a Legend**</font>

In [ ]:
from matplotlib.lines import Line2D

def export_bmi_hba1c_legend(
    out_path="bmi_hba1c_legend.png",
    dpi=300,
    bmi_color="#1f77b4",
    a1c_color="#d95f02"
):
    """
    Creates a thin, wide standalone legend for BMI & HbA1c.
    No axes, no frame — just symbols + labels on white background.
    """

    # Create a wide, low-height figure
    fig, ax = plt.subplots(figsize=(6, 0.6))  # adjust width as needed

    # Turn off all axes
    ax.axis("off")

    # Custom legend elements
    legend_elements = [
        Line2D([0], [0], color=bmi_color, marker="o", linestyle="-",
               markersize=6, label="BMI"),
        Line2D([0], [0], color=a1c_color, marker="s", linestyle="--",
               markersize=6, label="HbA1c")
    ]

    # Add the legend centered horizontally
    ax.legend(
        handles=legend_elements,
        loc="center",
        frameon=False,       # no border
        ncol=2,              # inline horizontally
        fontsize=10,
        handlelength=2.5,    # stretch lines slightly
        columnspacing=1.5
    )

    # Save the PNG
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)

    print(f"Legend exported to {out_path}")


In [ ]:
export_bmi_hba1c_legend("BMI_HbA1c_Legend.png")

##<font color="black">**eTable# Creation**</font>

Since not every patient has complete lab data for our study period, only a subset of patients with complete data was used for each cluster to construct the graphs. If readers are interested in those proportions, they can find them in the eTable that will be produced here.

In [ ]:
import pandas as pd
import numpy as np

def build_cluster_etable(groups,
                         cluster_counts,
                         sorted_cluster_patient_ids,
                         out_xlsx_path=None,
                         table_title=None):
    """
    Build an eTable summarizing per-group cluster sizes and
    BMI/HbA1c patient counts.

    Columns:
      - Group
      - Cluster (within-group rank, 1..n by descending size)
      - Total Patients
      - BMI Patient Count (%)
      - HbA1c Patient Count (%)

    Parameters
    ----------
    groups : dict
        Mapping group_name -> list of ORIGINAL cluster_ids.
        (e.g., Monotherapy / Dual Therapy / GLP-1 Therapy.)
    cluster_counts : dict
        Output from compute_age_adjusted_trends6:
          cluster_id -> {
             "cluster_total": int,
             "bmi_used": int,
             "a1c_used": int
          }
    sorted_cluster_patient_ids : dict
        Mapping cluster_id -> list of patient_ids, used to
        compute sizes and ranks within each group.
    out_xlsx_path : str or None
        If provided, writes the table to an Excel file using
        xlsxwriter. If None, just returns the DataFrame.
    table_title : str or None
        If provided, a title row will be added above the header
        and merged across all columns.
    """

    rows = []

    for group_name, cluster_ids in groups.items():
        # Rank clusters within this group by total patient count (desc)
        ordered = sorted(
            cluster_ids,
            key=lambda cid: len(sorted_cluster_patient_ids[cid]),
            reverse=True
        )

        for within_group_rank, cid in enumerate(ordered, start=1):
            counts = cluster_counts.get(cid, None)

            if counts is None:
                total_patients = len(sorted_cluster_patient_ids[cid])
                bmi_used = np.nan
                a1c_used = np.nan
            else:
                total_patients = counts.get("cluster_total", 0)
                bmi_used = counts.get("bmi_used", 0)
                a1c_used = counts.get("a1c_used", 0)

            if total_patients and total_patients > 0:
                bmi_pct = 100.0 * bmi_used / total_patients if bmi_used is not None else np.nan
                a1c_pct = 100.0 * a1c_used / total_patients if a1c_used is not None else np.nan
            else:
                bmi_pct = np.nan
                a1c_pct = np.nan

            if np.isnan(bmi_pct):
                bmi_display = ""
            else:
                bmi_display = f"{bmi_used} ({bmi_pct:.0f}%)"

            if np.isnan(a1c_pct):
                a1c_display = ""
            else:
                a1c_display = f"{a1c_used} ({a1c_pct:.0f}%)"

            rows.append({
                "Group": group_name,
                "Cluster": within_group_rank,      # group-specific rank
                "Total Patients": total_patients,
                "BMI Patient Count (%)": bmi_display,
                "HbA1c Patient Count (%)": a1c_display,
            })

    etable_df = pd.DataFrame(rows, columns=[
        "Group",
        "Cluster",
        "Total Patients",
        "BMI Patient Count (%)",
        "HbA1c Patient Count (%)"
    ])

    # Optional: write to Excel using xlsxwriter
    if out_xlsx_path is not None:
        with pd.ExcelWriter(out_xlsx_path, engine="xlsxwriter") as writer:
            # If we have a title, start the table at row 1; otherwise at row 0
            start_row = 1 if table_title is not None else 0

            etable_df.to_excel(
                writer,
                sheet_name="Cluster_eTable",
                index=False,
                startrow=start_row
            )

            workbook  = writer.book
            worksheet = writer.sheets["Cluster_eTable"]

            # ----- Title row (merged) -----
            if table_title is not None:
                n_cols = len(etable_df.columns)
                title_format = workbook.add_format({
                    "bold": True,
                    "font_size": 14,
                    "align": "center",
                    "valign": "vcenter"
                })
                # Merge from row 0, col 0 to row 0, col n_cols-1
                worksheet.merge_range(0, 0, 0, n_cols - 1, table_title, title_format)

            # ----- Header formatting -----
            header_format = workbook.add_format({
                "bold": True,
                "valign": "top",
                "border": 1
            })

            # Overwrite header row cells with format
            header_row = start_row  # row where headers are
            for col_num, value in enumerate(etable_df.columns.values):
                worksheet.write(header_row, col_num, value, header_format)

            # ----- Adjust column widths -----
            worksheet.set_column("A:A", 18)  # Group
            worksheet.set_column("B:B", 10)  # Cluster
            worksheet.set_column("C:C", 15)  # Total Patients
            worksheet.set_column("D:E", 22)  # BMI / HbA1c

    return etable_df

In [ ]:
!pip install xlsxwriter

In [ ]:
desired_order = [
    "Monotherapy",
    "Dual Therapy",
    "Complex Therapy",
    "Variant Therapy",
    "GLP-1 Therapy"
]

etable_groups = {
    gname: groups[gname]
    for gname in desired_order
    if gname in groups
}

etable_df = build_cluster_etable(
    groups=etable_groups,
    cluster_counts=cluster_counts,
    sorted_cluster_patient_ids=sorted_cluster_patient_ids,
    out_xlsx_path="Cluster_eTable_counts.xlsx",
    table_title="eTable X. Patient counts used for age-adjusted BMI and HbA1c trajectories by cluster group"
)

etable_df